# RVC Voice Server — automated multi-speaker training + inference
**Thesis:** Integrating Speech-to-Text and RVC-Based Voice Conversion into an AI Assistant
**Author:** Nguyen Hoang Ngoc Bao — 24MSE23204

This notebook replaces the old two-notebook workflow (`train_rvc.ipynb` + `serve_rvc.ipynb`, still kept for
reference) with a single always-on HTTP server that the app calls directly. Instead of manually editing a
`SPEAKER_NAME` cell and running the whole pipeline by hand for every user, the app now uploads samples and
triggers training automatically — this notebook just needs to be **running**.

**API contract** (matches `voice/rvc_client.py` in the app):
- `GET  /health` → `{"status": "ok"}`
- `GET  /models` → `{"speakers": [...]}` (speaker_ids with a trained model available)
- `POST /train` — multipart: `speaker_id` + one or more `files` (wav/mp3 samples) → enqueues a training job,
  returns immediately: `{"status": "queued", "message": "..."}`
- `GET  /train_status/<speaker_id>` → `{"status": "queued"|"running"|"done"|"failed", "message": "..."}`
- `POST /convert` — multipart: `audio` (mp3/wav) + `speaker_id` + `pitch` + `index_rate` → converted WAV bytes
- `DELETE /models/<speaker_id>` → deletes that speaker's `.pth`/`.index` from Drive and evicts it from the in-memory cache (called when a user deletes their voice profile in the app)
- `GET /models/<speaker_id>/download` → zipped `.pth`+`.index` for that speaker, so the app can keep its own local backup copy under `voice_storage/` (called right after training finishes)
- `POST /baseline/f5tts` — multipart: `gen_text` (required), `ref_audio` (wav, optional — defaults to the bundled reference clip), `ref_text`, `speed` → WAV bytes.
  Zero-shot baseline (hynt/F5-TTS-Vietnamese-ViVoice, F5-TTS) used for the RQ2 MOS comparison against the trained RVC voice, and optionally as the base-voice engine instead of edge-TTS (select it via a profile's `base_tts_voice = "f5tts:default"`).
- `POST /transcribe` — multipart: `audio` (any format ffmpeg/librosa can decode) + `language` (optional, e.g. `"vi"`) → `{"text": str, "language": str}`.
  Speech-to-Text via `vinai/PhoWhisper-large` (VinAI's Vietnamese fine-tune of Whisper, thesis Section 2.1) — the **input** half of the voice loop, everything else above is the **output** half. The app falls back to a local, CPU-only `openai-whisper` model (`voice/stt.py`) whenever this endpoint is unset/unreachable.

Only one T4 GPU is available, so training jobs run one at a time through an internal queue — submitting a
second speaker while one is training just waits its turn instead of failing.

**Before running:**
1. `Runtime > Change runtime type > GPU (T4)`
2. Run all cells top-to-bottom
3. Copy the tunnel URL printed at the end into the manager dashboard (`/`, after logging in at `/login`) → "🔌 Kết nối RVC (Colab)" section (saved via `POST /manager/rvc_config`)
4. Keep this notebook running — that's the one manual step. Everything after (sample upload → train → status →
   ready-to-speak) is automatic from the app side.


In [ ]:
# ── C. CLONE RVC + INSTALL DEPENDENCIES ──────────────────────────────────────
import os

if not os.path.exists("/content/RVC"):
    !git clone --depth=1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI /content/RVC 2>&1 | tail -5
else:
    print("RVC already cloned.")

%cd /content/RVC

# rvc-python (fairseq dependency) fails to build on this Colab Python version --
# same break already hit and fixed locally (see voice/rvc_local.py): fairseq's
# dataclass fields use a mutable-default pattern Python 3.11+ rejects at class-
# definition time. Cell G below shells out to the repo's own infer/cli.py
# instead, same as the local fallback, so rvc-python isn't needed at all. The
# repo's own requirments_*.txt files (note: no "e", that's the repo's own typo)
# pull in ~40 packages for the full Gradio webui + UVR5 vocal separation tool
# that aren't needed here and kept failing on mirror hash-mismatches/network
# stalls -- installing just what training/inference actually import instead.
!pip install -q av ffmpeg-python matplotlib praat-parselmouth scikit-learn tensorboard transformers faiss-cpu
!pip install -q pydub librosa soundfile flask

print("\n✓ Dependencies installed.")


In [ ]:
# ── C2. PATCH: REDUCE DATALOADER WORKERS (avoid Colab OOM) ──────────────────
# Confirmed for real: RVC-Project's train/train.py hardcodes num_workers=4,
# persistent_workers=True, prefetch_factor=8, pin_memory=True for its training
# DataLoader. On a free-tier Colab instance (2 vCPUs, ~12.7GB RAM -- the same
# limit the DataLoader's own runtime warning flags), 4 *persistent* worker
# processes each holding 8 prefetched batches of pinned (non-swappable) memory
# reliably exhausts RAM over a run. Confirmed for real: a live training run
# stalled hard at epoch 20 (one epoch took 7m41s vs ~4s for every other epoch --
# classic memory-pressure thrashing), then shortly after the epoch-40 checkpoint
# saved, the whole Colab kernel was OOM-killed and Jupyter auto-restarted it,
# silently dropping the in-progress training subprocess along with the Flask
# server and cloudflared tunnel. No CLI flag exposes this -- train/train.py
# hardcodes it, so it has to be source-patched after cloning, before any call
# to train_speaker() (cell F).
train_py = "/content/RVC/train/train.py"
with open(train_py, "r", encoding="utf-8") as f:
    _src = f.read()

_patched = _src.replace("num_workers=4,", "num_workers=2,").replace("prefetch_factor=8,", "prefetch_factor=2,")
if _patched == _src:
    raise RuntimeError("train/train.py DataLoader patch found nothing to replace -- upstream code may have changed since this was written, check manually.")

with open(train_py, "w", encoding="utf-8") as f:
    f.write(_patched)

print("✓ Patched train/train.py DataLoader: num_workers 4→2, prefetch_factor 8→2 (avoids Colab free-tier OOM).")


In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────
# Training hyperparameters + Drive paths shared by every speaker. Unlike the old
# train_rvc.ipynb, SPEAKER_NAME isn't fixed here -- train_speaker(speaker_id, ...)
# below takes it as a per-call argument (one call per guest via /train), not one
# edit-and-rerun-the-whole-notebook per speaker.
SAMPLE_RATE  = 40000     # RVC v2 standard: 40 kHz
TOTAL_EPOCHS = 200       # thesis target (Section 4.1 Stage 2)
BATCH_SIZE   = 8         # reduce to 4 if CUDA OOM on T4
SAVE_EVERY   = 20        # checkpoint every N epochs
F0_METHOD    = "rmvpe"   # RMVPE -- most accurate per thesis Section 2.3
RVC_VERSION  = "v2"
PITCH        = 0         # semitone shift default (0 = no change); /convert can override per-request
INDEX_RATE   = 0.75      # FAISS blend default (0=ignore index, 1=full retrieval); /convert can override
PROTECT      = 0.33      # protect voiceless consonants from over-conversion
SERVER_PORT  = 7860

# Google Drive paths -- trained models must persist here across runtime restarts
# (Colab wipes /content on every fresh session); pretrained assets under
# /content/RVC/assets do NOT need Drive since they re-download from Hugging Face
# each run instead (next cell), identically for every speaker.
DRIVE_ROOT = "/content/drive/MyDrive/rvc_training"
MODELS_DIR = f"{DRIVE_ROOT}/models"

print("Configuration:")
print(f"  models_dir  : {MODELS_DIR}")
print(f"  sample_rate : {SAMPLE_RATE} Hz")
print(f"  epochs      : {TOTAL_EPOCHS}")
print(f"  batch_size  : {BATCH_SIZE}")
print(f"  f0_method   : {F0_METHOD}")
print(f"  rvc_version : {RVC_VERSION}")
print(f"  server_port : {SERVER_PORT}")


In [ ]:
# ── B. MOUNT GOOGLE DRIVE ────────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount("/content/drive")
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Drive mounted. Trained models are stored under: {MODELS_DIR}")


In [ ]:
# ── D. GPU CHECK ─────────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU (T4) and reconnect."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU  : {gpu_name}")
print(f"  VRAM : {vram_gb:.1f} GB")
if vram_gb < 10:
    print("⚠️ Low VRAM — consider reducing BATCH_SIZE to 4.")


In [ ]:
# ── E. DOWNLOAD PRETRAINED V2 MODELS ─────────────────────────────────────────────────────────
import os, urllib.request, zipfile

HF_BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"
# HuBERT moved from a single fairseq-format assets/hubert/hubert_base.pt to a
# HuggingFace transformers-format directory (confirmed against the current repo's
# own infer/hubert.py: HUBERT_MODEL_PATH = assets/hubert_base/, loaded via
# transformers, expects config.json + preprocessor_config.json + pytorch_model.bin).
ASSETS = {
    "assets/pretrained_v2/f0G40k.pth":              f"{HF_BASE}/pretrained_v2/f0G40k.pth",
    "assets/pretrained_v2/f0D40k.pth":               f"{HF_BASE}/pretrained_v2/f0D40k.pth",
    "assets/hubert_base/config.json":                f"{HF_BASE}/hubert_base/config.json",
    "assets/hubert_base/preprocessor_config.json":   f"{HF_BASE}/hubert_base/preprocessor_config.json",
    "assets/hubert_base/pytorch_model.bin":          f"{HF_BASE}/hubert_base/pytorch_model.bin",
    "assets/rmvpe/rmvpe.pt":                         f"{HF_BASE}/rmvpe.pt",
}

for dest, url in ASSETS.items():
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest):
        print(f"  ✓ {os.path.basename(dest)} (cached)")
    else:
        print(f"  Downloading {os.path.basename(dest)} …")
        urllib.request.urlretrieve(url, dest)
        print(f"  ✓ {os.path.basename(dest)}  ({os.path.getsize(dest)/1e6:.0f} MB)")

print("\n✓ All pretrained assets ready.")


## Training pipeline (parameterized by `speaker_id`)
Same steps as the original `train_rvc.ipynb` (slice/normalize → preprocess → F0 → HuBERT features → train →
FAISS index → export), just wrapped in a function instead of hardcoded to one `SPEAKER_NAME` global, so the
worker thread below can run it for any speaker_id submitted by the app.

In [ ]:
# ── F. TRAINING PIPELINE FUNCTION ────────────────────────────────────────────
import glob, os, queue, re, subprocess, sys, shutil, threading
import numpy as np
import faiss
from pydub import AudioSegment
from pydub.silence import split_on_silence


def _slice_and_normalize(src: str, out_dir: str, sr: int = SAMPLE_RATE,
                          min_ms: int = 3000, max_ms: int = 8000) -> int:
    audio = AudioSegment.from_file(src)
    audio = audio.set_frame_rate(sr).set_channels(1)
    audio = audio.apply_gain(-20.0 - audio.dBFS)

    chunks = split_on_silence(
        audio, min_silence_len=300, silence_thresh=audio.dBFS - 16, keep_silence=150,
    )

    stem = os.path.splitext(os.path.basename(src))[0]
    saved, buf = 0, AudioSegment.empty()
    for chunk in chunks:
        buf += chunk
        while len(buf) >= min_ms:
            seg = buf[:max_ms]
            buf = buf[max_ms:]
            seg.export(os.path.join(out_dir, f"{stem}_{saved:04d}.wav"), format="wav")
            saved += 1
    if len(buf) >= min_ms:
        buf.export(os.path.join(out_dir, f"{stem}_{saved:04d}.wav"), format="wav")
        saved += 1
    return saved


# Colab's own kernel environment has been observed with a malformed PYTHONHASHSEED
# (confirmed for real: subprocess.run() inheriting it verbatim crashed every training
# subprocess below at Python interpreter bootstrap, before any of their own code ran,
# with "Fatal Python error: config_init_hash_seed: PYTHONHASHSEED must be 'random' or
# an integer in range [0; 4294967295]"). Force a valid value in a copy of the
# environment rather than trusting whatever this kernel process happens to have --
# same pattern convert_speaker() below already uses for its own env fix (PYTHONUTF8).
#
# Also confirmed for real: running "python train/preprocess.py" (or any of the other
# train/... scripts below) as a subprocess fails with "ModuleNotFoundError: No module
# named 'infer'" -- Python only auto-adds a script's *own* directory (train/) to
# sys.path, not the repo root two levels up that infer/, train/, configs/, i18n/ all
# live under, so `from infer.audio import load_audio` (train/preprocess.py's own first
# import) can't resolve. webui.py's Gradio process presumably gets this for free by
# being *itself* launched from the repo root; a bare subprocess.run() of a nested
# script gets no such thing automatically. PYTHONPATH fixes it for every script below.
_CHILD_ENV = os.environ.copy()
_CHILD_ENV["PYTHONHASHSEED"] = "0"
_CHILD_ENV["PYTHONPATH"] = "/content/RVC"  # same repo root train_speaker() itself os.chdir()s to

# Early stopping for the main training subprocess -- ported from voice/rvc_local.py's
# already-proven _run_training_with_early_stop()/_finalize_from_checkpoint() (same
# RVC-Project train/train.py, same small personal-voice-sample dataset shape, same
# log_interval=200 default). Confirmed for real on this Colab pipeline that running
# the full TOTAL_EPOCHS unconditionally costs real GPU time (and Drive storage, once
# every checkpoint is synced) well past the point the generator loss stops improving --
# stopping early once it plateaus saves both.
#
# RVC's generator loss (loss_gen + loss_fm + loss_mel + loss_kl -- train.py's own
# "loss_gen_all" minus the discriminator's own adversarial loss_disc, which reflects
# the discriminator's state rather than output quality) is noisy epoch-to-epoch since
# this is adversarial (GAN) training, not a monotonically-decreasing supervised loss --
# patience needs to be generous enough to ride out normal oscillation instead of
# bailing on a temporary bad epoch. Values match rvc_local.py's own tuning notes:
#   - EARLY_STOP_MIN_EPOCHS: no stopping before this many epochs -- the first several
#     are the noisiest (generator/discriminator still finding balance).
#   - EARLY_STOP_PATIENCE: 10, not 5 -- 5 was tried and rejected there: with small
#     personal-voice datasets (a handful of minutes of audio, few steps/epoch) the
#     loss estimate per epoch is itself noisy, and 5 non-improving epochs is well
#     within normal GAN oscillation, risking stopping right before a real improvement.
#   - EARLY_STOP_MIN_DELTA: minimum loss decrease to count as "improvement" -- without
#     this, floating-point-noise-sized "improvements" would keep resetting patience.
EARLY_STOP_MIN_EPOCHS = 15
EARLY_STOP_PATIENCE    = 10
EARLY_STOP_MIN_DELTA   = 0.01
# Generous enough to ride out a slow checkpoint save (observed for real: up to ~2min
# on this Colab+Drive setup at epochs 20/40/60) without false-triggering as a hang.
STALL_TIMEOUT_SEC = 600

# Matches train/train.py's own per-step log line (see its logger.info(f"loss_disc=...
# loss_gen=... loss_fm=...loss_mel=... loss_kl=...")) -- \s* rather than a fixed space
# count since that f-string has inconsistent spacing around the commas.
_LOSS_LINE_RE = re.compile(
    r"loss_gen=([\d.]+),\s*loss_fm=([\d.]+),\s*loss_mel=([\d.]+),\s*loss_kl=([\d.]+)"
)
# train/train.py's per-epoch marker (i18n("====> ...").format(epoch, ...)) -- the
# "====> " prefix is part of the i18n *key* itself, kept as-is by every locale file
# checked (including en_US: "====> Epoch: {} {}"), so matching just the prefix reliably
# detects an epoch boundary regardless of which language train.py's logger is using.
_EPOCH_MARKER = "====> "


def _run_training_with_early_stop(exp_dir: str, speaker_id: str, epochs: int, report) -> int:
    """
    Runs train/train.py, watching its live output so training can stop early once the
    generator loss hasn't improved for EARLY_STOP_PATIENCE epochs, instead of always
    running the full epoch count. Returns the last epoch number actually reached
    (< epochs if stopped early). Raises RuntimeError on a real (non-early-stop) failure.
    """
    proc = subprocess.Popen([
        sys.executable, "-m", "train.train",
        "-e", speaker_id, "-sr", "40k", "-f0", "1", "-bs", str(BATCH_SIZE),
        "-g", "0", "-te", str(epochs), "-se", str(SAVE_EVERY),
        "-pg", "assets/pretrained_v2/f0G40k.pth", "-pd", "assets/pretrained_v2/f0D40k.pth",
        "-l", "1", "-c", "0", "-sw", "0", "-v", RVC_VERSION,
    ], env=_CHILD_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    # `for line in proc.stdout` blocks with no timeout, so a hung (rather than crashed)
    # child would block this forever with no way to notice. Reading on a background
    # thread and pulling from a queue with a timeout is the standard way to get a
    # timeout on a blocking pipe read in Python (no cross-platform way to put a timeout
    # directly on the read itself) -- same pattern as voice/rvc_local.py's own version.
    lines = queue.Queue()

    def _pump():
        try:
            for line in proc.stdout:
                lines.put(line)
        finally:
            lines.put(None)  # sentinel: stdout closed (process exited)

    threading.Thread(target=_pump, daemon=True).start()

    epoch           = 0
    last_metric     = None
    metric_is_fresh = False  # a NEW loss line arrived since the last epoch boundary
    best_metric     = None
    best_epoch      = 0
    no_improve      = 0
    stopped_early   = False
    stalled         = False

    try:
        while True:
            try:
                line = lines.get(timeout=STALL_TIMEOUT_SEC)
            except queue.Empty:
                report(f"No output for {STALL_TIMEOUT_SEC // 60} minutes -- assuming the "
                       f"training process is stuck and stopping it.")
                stalled = True
                proc.terminate()
                break

            if line is None:  # stdout closed -- process has exited
                break

            m = _LOSS_LINE_RE.search(line)
            if m:
                last_metric = sum(float(g) for g in m.groups())
                metric_is_fresh = True
                continue
            if _EPOCH_MARKER not in line:
                continue

            epoch += 1
            # train.py logs a loss line every train.log_interval *steps* (200 by
            # default), not every epoch -- on a small personal-voice dataset a fresh
            # reading can be many epochs apart. Only counting epochs with a genuinely
            # fresh reading (not re-evaluating a stale one) avoids a false-early-stop
            # from the same loss value being counted as "no improvement" repeatedly.
            if metric_is_fresh:
                if best_metric is None or last_metric < best_metric - EARLY_STOP_MIN_DELTA:
                    best_metric, best_epoch, no_improve = last_metric, epoch, 0
                else:
                    no_improve += 1
                report(f"Epoch {epoch}/{epochs} \u2014 loss {last_metric:.3f} "
                       f"(best {best_metric:.3f} @ epoch {best_epoch}, "
                       f"{no_improve}/{EARLY_STOP_PATIENCE} without improvement)")
                metric_is_fresh = False

            if epoch >= EARLY_STOP_MIN_EPOCHS and no_improve >= EARLY_STOP_PATIENCE:
                report(f"No improvement for {EARLY_STOP_PATIENCE} epochs -- stopping early "
                       f"at epoch {epoch} (target was {epochs}) to save time and storage.")
                stopped_early = True
                proc.terminate()
                break
    finally:
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()

    if stalled:
        raise RuntimeError(
            f"Training stalled at epoch {epoch} -- no output for {STALL_TIMEOUT_SEC // 60} "
            f"minutes, process was terminated."
        )
    if not stopped_early and proc.returncode != 0:
        raise RuntimeError(f"Training failed (exit code {proc.returncode}) -- check Colab logs above.")

    return epoch


def _finalize_from_checkpoint(exp_dir: str, speaker_id: str, epoch_reached: int):
    """
    Builds assets/weights/<speaker_id>.pth from the latest raw training checkpoint
    when early stopping cut the run short. train/train.py's own savee() call (see
    train/process_ckpt.py) only fires once epoch >= the full -te target, which an
    early-stopped run never reaches, so we have to reproduce that step ourselves.
    """
    g_checkpoints = sorted(glob.glob(os.path.join(exp_dir, "G_*.pth")),
                            key=os.path.getmtime, reverse=True)
    if not g_checkpoints:
        raise RuntimeError("Early-stopped before any checkpoint was saved -- nothing to finalize.")

    config_path = os.path.join(exp_dir, "config.json")
    snippet = f"""
import json, torch
from train.utils import HParams
from train.process_ckpt import savee

hps = HParams(**json.load(open(r"{config_path}", encoding="utf-8")))
ckpt = torch.load(r"{g_checkpoints[0]}", map_location="cpu", weights_only=False)["model"]
result = savee(ckpt, {SAMPLE_RATE}, 1, "{speaker_id}", {epoch_reached}, "{RVC_VERSION}", hps)
print("SAVEE_RESULT:", result)
"""
    result = subprocess.run([sys.executable, "-c", snippet], env=_CHILD_ENV,
                             capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Finalizing early-stopped model crashed:\n{result.stderr[-1500:]}")
    if "Traceback" in result.stdout:
        # savee() catches its own exceptions and returns the traceback as a string
        # rather than raising -- see train/process_ckpt.py.
        raise RuntimeError(f"savee() failed while finalizing early-stopped model:\n{result.stdout[-1500:]}")


def _run_step(cmd, label):
    result = subprocess.run(cmd, capture_output=True, text=True, env=_CHILD_ENV)
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed:\n{result.stderr[-1500:]}")
    return result.stdout


def train_speaker(speaker_id: str, raw_dir: str, progress_cb=None):
    """
    Full pipeline for one speaker: slice/normalize -> preprocess -> F0 ->
    HuBERT features -> train RVC v2 -> FAISS index -> export to Drive.
    raw_dir must already contain the uploaded raw wav/mp3 samples.
    """
    def report(msg):
        print(f"[{speaker_id}] {msg}")
        if progress_cb:
            progress_cb(msg)

    os.chdir("/content/RVC")
    # savee() (train/process_ckpt.py) writes to the hardcoded relative path
    # "assets/weights/<name>.pth" and does NOT create its parent dir -- confirmed
    # for real: a fresh --depth=1 clone of the repo does not ship this folder, so
    # torch.save() inside savee() fails with "RuntimeError: Parent directory
    # assets/weights does not exist." This is deterministic, not flaky -- it broke
    # on the *first* full 200-epoch run this pipeline ever completed, silently (the
    # exception is swallowed by savee()'s own bare except and only surfaces as a
    # logged traceback, never raised), so "training complete" left no usable model.
    # voice/rvc_local.py hit this exact bug first and already carries the same fix.
    os.makedirs("assets/weights", exist_ok=True)
    sliced_dir = f"/content/dataset_{speaker_id}_sliced"
    os.makedirs(sliced_dir, exist_ok=True)
    exp_dir = f"logs/{speaker_id}"
    os.makedirs(exp_dir, exist_ok=True)

    report("Slicing & normalizing samples…")
    sources = (glob.glob(os.path.join(raw_dir, "*.wav")) +
               glob.glob(os.path.join(raw_dir, "*.mp3")) +
               glob.glob(os.path.join(raw_dir, "*.webm")) +
               glob.glob(os.path.join(raw_dir, "*.ogg")) +
               glob.glob(os.path.join(raw_dir, "*.m4a")))
    if not sources:
        raise RuntimeError(f"No audio files found in {raw_dir}")
    total_segs = sum(_slice_and_normalize(f, sliced_dir) for f in sources)
    report(f"{total_segs} segments produced.")
    if total_segs < 20:
        report("⚠️ Few segments — quality may be lower than ideal, continuing anyway.")

    # Script paths/arg formats below match the *current* RVC-Project main branch,
    # confirmed directly against its source (not the old root-level scripts this
    # pipeline originally shelled out to, which the upstream repo has since removed
    # entirely -- see train_speaker()'s own docstring history for what broke and why).
    # Run every RVC-repo script below as "-m package.module", not a bare file path --
    # confirmed for real that "python train/preprocess.py" fails with a circular-import-
    # looking error ("cannot import name 'utils' from partially initialized module
    # 'train'"). Root cause: Python auto-adds a *bare-file-path* script's own directory
    # to sys.path[0] -- with train/preprocess.py that's "train/" itself, which then
    # collides with the top-level "train" *package* our PYTHONPATH fix (above) also
    # makes resolvable (train/train.py becomes ambiguous with the train/ package it's
    # inside). "-m" mode has no such auto-add -- it resolves "train.preprocess" etc.
    # as ordinary dotted imports via PYTHONPATH/cwd only, so there's nothing to collide.
    report("Preprocessing…")
    _run_step([sys.executable, "-m", "train.preprocess",
               sliced_dir, str(SAMPLE_RATE), "4", exp_dir, "False", "3.7"], "Preprocessing")

    report("Extracting F0 (RMVPE)…")
    # train/dataset/extract_f0.py argv (cuda branch): cuda <n_part> <i_part> <i_gpu> <exp_dir> <is_half>
    _run_step([sys.executable, "-m", "train.dataset.extract_f0",
               "cuda", "1", "0", "0", exp_dir, "False"], "F0 extraction")

    report("Extracting HuBERT features…")
    # train/dataset/extract_hubert_feature.py argv (7-arg/GPU branch):
    # <device> <n_part> <i_part> <i_gpu> <exp_dir> <version> <is_half>
    _run_step([sys.executable, "-m", "train.dataset.extract_hubert_feature",
               "cuda:0", "1", "0", "0", exp_dir, RVC_VERSION, "False"], "Feature extraction")

    report("Writing training config…")
    # train/train.py (via train/utils.py get_hparams()) now requires a config.json in
    # exp_dir *before* it runs -- webui.py generates this from a template on every run;
    # our pipeline never needed to before because the old train.py didn't require one.
    # sr=40k always uses the v1-shaped config template regardless of RVC_VERSION (confirmed
    # against webui.py's own selection logic: `if version19 == "v1" or sr2 == "40k"`) --
    # already present locally, no download needed, since the whole repo (incl. configs/)
    # was git-cloned in cell C.
    shutil.copy2("configs/v1/40k.json", os.path.join(exp_dir, "config.json"))

    report("Building training filelist…")
    gt_wavs_dir = os.path.join(exp_dir, "0_gt_wavs")
    feature_dir = os.path.join(exp_dir, "3_feature768")
    f0_dir      = os.path.join(exp_dir, "2a_f0")
    f0nsf_dir   = os.path.join(exp_dir, "2b-f0nsf")

    wav_map = {os.path.splitext(os.path.basename(w))[0]: w
               for w in sorted(glob.glob(os.path.join(gt_wavs_dir, "*.wav")))}
    feat_map = {os.path.splitext(os.path.basename(f))[0]: f
                for f in sorted(glob.glob(os.path.join(feature_dir, "*.npy")))}
    common = sorted(set(wav_map) & set(feat_map))
    if not common:
        raise RuntimeError("No matching wav/feature pairs — preprocessing or feature extraction failed.")

    lines = [f"{wav_map[s]}|{feat_map[s]}|{os.path.join(f0_dir, s + '.wav.npy')}|"
             f"{os.path.join(f0nsf_dir, s + '.wav.npy')}|0" for s in common]
    # No silence-padding "mute" lines here -- webui.py's own GUI flow adds them, but
    # train/data_utils.py's dataset loader has no code-level dependency on their presence
    # (checked directly: no length/count assertions, no special-casing), and
    # voice/rvc_local.py's already-working local training pipeline (same current
    # RVC-Project API, independently re-derived) skips them too. Matching that proven
    # approach rather than carrying an extra download + untested assumption.
    filelist_path = os.path.join(exp_dir, "filelist.txt")
    with open(filelist_path, "w") as fh:
        fh.write("\n".join(lines))
    report(f"Filelist: {len(lines)} entries.")

    report(f"Training up to {TOTAL_EPOCHS} epochs (this takes ~30-60 min on a T4; "
           f"may stop early once the generator loss plateaus)…")
    reached_epoch = _run_training_with_early_stop(exp_dir, speaker_id, TOTAL_EPOCHS, report)
    report(f"Training stopped at epoch {reached_epoch} (target was {TOTAL_EPOCHS}).")

    report("Building FAISS index…")
    feat_files = sorted(glob.glob(os.path.join(feature_dir, "*.npy")))
    features = np.concatenate([np.load(f) for f in feat_files], axis=0).astype("float32")
    n_ivf = max(min(int(16 * np.sqrt(len(features))), len(features) // 39 + 1), 4)
    index = faiss.index_factory(features.shape[1], f"IVF{n_ivf},Flat")
    index.train(features)
    index.add(features)
    index_path = os.path.join(exp_dir, f"{speaker_id}.index")
    faiss.write_index(index, index_path)

    # The extracted, inference-ready model (half-precision weights, encoder-q dropped,
    # config embedded) is what train/train.py's own savee() call writes at the end of
    # training -- confirmed against train/process_ckpt.py: always exactly
    # assets/weights/{name}.pth, unconditionally, once epoch >= total_epoch, regardless of
    # the -sw/save_every_weights flag (that flag only gates *intermediate* per-epoch
    # extracts, not this final one). This is the file to export, not the raw G_*.pth/
    # D_*.pth training checkpoints left behind in exp_dir (those still include optimizer
    # state and encoder-q weights -- full training state, not a distributable model).
    pth_path = f"assets/weights/{speaker_id}.pth"
    if not os.path.exists(pth_path):
        report("Building final model from the last checkpoint (early-stopped run)…")
        _finalize_from_checkpoint(exp_dir, speaker_id, reached_epoch)
    if not os.path.exists(pth_path):
        raise RuntimeError(f"No extracted model found at {pth_path} after training.")

    out_pth   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    out_index = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    shutil.copy2(pth_path, out_pth)
    shutil.copy2(index_path, out_index)
    report(f"Exported model → {out_pth}")

    shutil.rmtree(sliced_dir, ignore_errors=True)
    return out_pth, out_index


print("✓ train_speaker() ready.")


In [ ]:
# ── G. CONVERT (subprocess-based, same approach as the local fallback --
#      rvc-python/fairseq doesn't build on this Python version, see cell C) ──
import subprocess, sys

F0_METHOD_CONVERT = "rmvpe"


def has_trained_model(speaker_id: str) -> bool:
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    return os.path.exists(pth_path) and os.path.exists(index_path)


def convert_speaker(speaker_id: str, in_path: str, out_path: str, pitch: int, index_rate: float):
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    child_env = os.environ.copy()
    child_env["PYTHONUTF8"] = "1"  # infer/cli.py prints non-Latin status text -- avoid UnicodeEncodeError
    result = subprocess.run([
        sys.executable, "-m", "infer.cli",
        "--model", pth_path,
        "--index", index_path,
        "--input", in_path,
        "--output", out_path,
        "--pitch", str(pitch),
        "--f0-method", F0_METHOD_CONVERT,
        "--index-rate", str(index_rate),
        "--protect", str(PROTECT),
        "--overwrite",
    ], cwd="/content/RVC", capture_output=True, timeout=120, env=child_env,
       encoding="utf-8", errors="replace")
    if result.returncode != 0:
        raise RuntimeError((result.stdout[-800:] + "\n" + result.stderr[-800:]).strip())


def list_trained_speakers():
    return sorted({
        os.path.splitext(os.path.basename(p))[0]
        for p in glob.glob(os.path.join(MODELS_DIR, "*.pth"))
    })


print("✓ Convert function ready (subprocess-based). Trained speakers so far:", list_trained_speakers())


In [ ]:
# ── H. TRAINING JOB QUEUE (one GPU → one job at a time) ──────────────────────
import queue, threading, shutil

TRAIN_JOBS = {}   # speaker_id -> {"status": "queued"|"running"|"done"|"failed", "message": str}
_train_queue = queue.Queue()
_jobs_lock = threading.Lock()


def _set_job(speaker_id, **kwargs):
    with _jobs_lock:
        TRAIN_JOBS.setdefault(speaker_id, {}).update(kwargs)


def _worker_loop():
    while True:
        speaker_id, raw_dir = _train_queue.get()
        _set_job(speaker_id, status="running", message="Bat dau huan luyen...")
        try:
            train_speaker(speaker_id, raw_dir, progress_cb=lambda m: _set_job(speaker_id, message=m))
            _set_job(speaker_id, status="done", message="Huan luyen hoan tat.")
        except Exception as exc:
            print(f"[{speaker_id}] TRAINING FAILED: {exc}")
            _set_job(speaker_id, status="failed", message=str(exc))
        finally:
            shutil.rmtree(raw_dir, ignore_errors=True)
            _train_queue.task_done()


threading.Thread(target=_worker_loop, daemon=True).start()
print("✓ Training worker thread started (processes one speaker at a time).")


## STT Lab Tier 2: LoRA fine-tune Whisper (tiny/base only)

Serves the same guest-facing STT Lab as clone-voice-station's own local
fallback (voice/stt_local_train.py) -- a guest can pick "Colab" as the
training backend to compare it against "Local". Runs on the same GPU as the
RVC pipeline above, through its own separate job queue (not merged into
`_train_queue`) -- keep in mind an RVC job and an STT job still compete for
the same physical GPU if you run both at once.

In [ ]:
!pip install -q peft

In [ ]:
# ── N. STT LORA TRAINING PIPELINE FUNCTION (Tier 2) ────────────────────────────
import json
import os
import tempfile
import zipfile
import librosa
import torch
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

STT_ADAPTERS_DIR = "/content/stt_adapters"
os.makedirs(STT_ADAPTERS_DIR, exist_ok=True)

_STT_HF_MODEL_BY_NAME = {"whisper-tiny": "openai/whisper-tiny", "whisper-base": "openai/whisper-base"}
STT_EPOCHS = 3
STT_LEARNING_RATE = 1e-3
STT_LORA_R = 8
STT_LORA_ALPHA = 32
STT_LORA_DROPOUT = 0.05
STT_LORA_TARGET_MODULES = ["q_proj", "v_proj"]


def train_stt_adapter(adapter_id, raw_dir, base_model, transcripts, resume_zip_path=None, progress_cb=None):
    """
    raw_dir: directory of uploaded audio files (see /stt_train route below).
    transcripts: {filename: reference_text}.
    resume_zip_path: path to an uploaded resume-adapter zip (adapter_model.safetensors +
                      adapter_config.json), or None to start fresh.
    Returns the output directory holding the trained adapter.
    """
    hf_model_name = _STT_HF_MODEL_BY_NAME.get(base_model)
    if not hf_model_name:
        raise ValueError(f"Unsupported base_model: {base_model}")

    def _report(msg):
        print(f"[{adapter_id}] {msg}")
        if progress_cb:
            progress_cb(msg)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    _report(f"Dang tai model nen {hf_model_name} ({device})...")
    processor = WhisperProcessor.from_pretrained(hf_model_name, language="vietnamese", task="transcribe")
    base = WhisperForConditionalGeneration.from_pretrained(hf_model_name)
    base.generation_config.language = "vietnamese"
    base.generation_config.task = "transcribe"

    resume_dir = None
    if resume_zip_path:
        resume_dir = tempfile.mkdtemp(prefix=f"resume_{adapter_id}_")
        with zipfile.ZipFile(resume_zip_path) as zf:
            zf.extractall(resume_dir)

    if resume_dir:
        _report("Tiep tuc huan luyen tu adapter da co...")
        model = PeftModel.from_pretrained(base, resume_dir, is_trainable=True)
    else:
        lora_config = LoraConfig(r=STT_LORA_R, lora_alpha=STT_LORA_ALPHA,
                                  target_modules=STT_LORA_TARGET_MODULES, lora_dropout=STT_LORA_DROPOUT)
        model = get_peft_model(base, lora_config)
    model.to(device)
    model.train()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=STT_LEARNING_RATE)
    use_amp = device == "cuda"
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    filenames = [f for f in os.listdir(raw_dir) if f in transcripts]
    n = len(filenames)
    for epoch in range(STT_EPOCHS):
        for i, fname in enumerate(filenames):
            audio, _sr = librosa.load(os.path.join(raw_dir, fname), sr=16000, mono=True)
            input_features = processor.feature_extractor(
                audio, sampling_rate=16000, return_tensors="pt"
            ).input_features.to(device)
            labels = processor.tokenizer(transcripts[fname], return_tensors="pt").input_ids.to(device)

            optimizer.zero_grad()
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    loss = model(input_features=input_features, labels=labels).loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = model(input_features=input_features, labels=labels).loss
                loss.backward()
                optimizer.step()
            _report(f"Epoch {epoch + 1}/{STT_EPOCHS}, mau {i + 1}/{n}, loss={loss.item():.3f}")

    output_dir = os.path.join(STT_ADAPTERS_DIR, str(adapter_id))
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    _report("Da luu adapter.")
    return output_dir

In [ ]:
# ── O. STT TRAINING JOB QUEUE (Tier 2 -- separate from RVC's _train_queue) ────
# Same GPU as the RVC pipeline above, so an RVC job and an STT job still
# compete for it if run at the same time -- kept as two queues rather than
# merged into one scheduler since this is a comparison/demo feature, not a
# hardened production one.
STT_JOBS = {}   # adapter_id -> {"status": "queued"|"running"|"done"|"failed", "message": str}
_stt_train_queue = queue.Queue()
_stt_jobs_lock = threading.Lock()


def _set_stt_job(adapter_id, **kwargs):
    with _stt_jobs_lock:
        STT_JOBS.setdefault(adapter_id, {}).update(kwargs)


def _stt_worker_loop():
    while True:
        adapter_id, raw_dir, base_model, transcripts, resume_zip_path = _stt_train_queue.get()
        _set_stt_job(adapter_id, status="running", message="Bat dau huan luyen...")
        try:
            train_stt_adapter(
                adapter_id, raw_dir, base_model, transcripts, resume_zip_path=resume_zip_path,
                progress_cb=lambda m: _set_stt_job(adapter_id, message=m),
            )
            _set_stt_job(adapter_id, status="done", message="Huan luyen hoan tat.")
        except Exception as exc:
            print(f"[{adapter_id}] STT TRAINING FAILED: {exc}")
            _set_stt_job(adapter_id, status="failed", message=str(exc))
        finally:
            shutil.rmtree(raw_dir, ignore_errors=True)
            _stt_train_queue.task_done()


threading.Thread(target=_stt_worker_loop, daemon=True).start()
print("Da khoi dong STT training worker thread (xu ly 1 adapter/lan).")

## Baseline: F5-TTS-Vietnamese-ViVoice (zero-shot voice cloning)
Adds `hynt/F5-TTS-Vietnamese-ViVoice` — a Vietnamese fine-tune of F5-TTS (flow-matching,
trained on ~1000h of Vietnamese speech, ViVoice dataset) — as a second **zero-shot**
baseline next to XTTS-v2 for the thesis's RQ2 comparison (does per-speaker RVC training
beat zero-shot cloning on naturalness / speaker similarity?). Unlike RVC, it needs no
per-speaker training: it clones a voice from a single ~6-10s reference clip + its exact
transcript, supplied at inference time.

It is also wired in as an optional **base-voice engine** in the app (`voice/tts.py`),
selectable in place of edge-TTS by setting a profile's `base_tts_voice` to `"f5tts:default"`
— useful for demoing whether a more natural Vietnamese base signal improves the final
RVC-converted output.

**Before this section is usable:** upload one clean reference clip to
`f5tts_assets/reference.wav` under `DRIVE_ROOT` and set `REF_TEXT_F5TTS` (next cell) to
its exact transcript — F5-TTS has no default voice of its own, it always clones from a
reference.

In [ ]:
# ── L1. INSTALL F5-TTS-VIETNAMESE-VIVOICE ────────────────────────────────────
import os

F5TTS_DIR = "/content/F5-TTS-Vietnamese"
if not os.path.exists(F5TTS_DIR):
    !git clone --depth=1 https://github.com/nguyenthienhy/F5-TTS-Vietnamese {F5TTS_DIR} 2>&1 | tail -5
else:
    print("F5-TTS-Vietnamese already cloned.")

%cd {F5TTS_DIR}
!pip install -q -e .
%cd /content/RVC

# Verify the editable install actually resolved in *this* kernel before moving on --
# confirmed for real that `pip install -e .` can report success while `import f5_tts`
# still fails in the same already-running session (a known Jupyter/Colab gotcha with
# editable installs of src-layout packages -- F5-TTS-Vietnamese's own pyproject.toml has
# no explicit [tool.setuptools.packages.find] for its src/ layout). A manual sys.path
# entry sidesteps that instead of requiring a runtime restart.
import sys
try:
    import f5_tts  # noqa: F401
    print('F5-TTS-Vietnamese installed (import OK).')
except ModuleNotFoundError:
    src_dir = f'{F5TTS_DIR}/src'
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import f5_tts  # noqa: F401  -- raises again here with a clear traceback if this fallback also fails
    print(f'F5-TTS-Vietnamese installed (import OK via sys.path fallback: {src_dir}).')


In [ ]:
# ── L2. DOWNLOAD hynt/F5-TTS-Vietnamese-ViVoice CHECKPOINT ───────────────────
# Three-tier source, cheapest/most-reliable first:
#   1. Already cached under DRIVE_ROOT from a prior session -- no network at all.
#   2. A pre-uploaded Google Drive mirror (gdown) -- faster than HF and doesn't
#      depend on HF's own reachability, but depends on this folder staying shared:
#      https://drive.google.com/drive/folders/1okDsldsKkALbFAzGCuGFlwohju0bXxxu
#   3. HuggingFace itself (hf_hub_download, the original source) -- slower and was
#      seen to be flaky for a ~5GB file on Colab's network, but always available
#      regardless of whether the Drive mirror above still exists.
import os

F5TTS_REPO        = "hynt/F5-TTS-Vietnamese-ViVoice"
F5TTS_ASSETS_DIR  = f"{DRIVE_ROOT}/f5tts_assets"
F5TTS_DRIVE_FOLDER_ID = "1okDsldsKkALbFAzGCuGFlwohju0bXxxu"
os.makedirs(F5TTS_ASSETS_DIR, exist_ok=True)

F5TTS_CKPT  = os.path.join(F5TTS_ASSETS_DIR, "model_last.pt")
F5TTS_VOCAB = os.path.join(F5TTS_ASSETS_DIR, "vocab.txt")

def _have_both():
    return os.path.exists(F5TTS_CKPT) and os.path.exists(F5TTS_VOCAB)

if _have_both():
    print("✓ Already cached in Drive from a prior session -- skipping download.")
else:
    print("Not cached yet -- trying the Drive mirror first (gdown)...")
    try:
        get_ipython().system('pip install -q gdown')
        import gdown
        # Also fetches a .cache archive alongside the two files this pipeline
        # actually needs -- harmless, just not used below.
        gdown.download_folder(id=F5TTS_DRIVE_FOLDER_ID, output=F5TTS_ASSETS_DIR, quiet=False, use_cookies=False)
    except Exception as e:
        print(f"Drive mirror download failed or incomplete ({e}) -- falling back to HuggingFace.")

    if not _have_both():
        print("Falling back to HuggingFace (hf_hub_download)...")
        from huggingface_hub import hf_hub_download, list_repo_files

        _repo_files = list_repo_files(F5TTS_REPO)
        print("Files in repo:", _repo_files)

        def _pick(suffixes):
            for suf in suffixes:
                for fname in _repo_files:
                    if fname.endswith(suf):
                        return fname
            return None

        _ckpt_name  = _pick([".pt", ".safetensors"]) or "model_last.pt"
        _vocab_name = _pick(["vocab.txt"])

        if not os.path.exists(F5TTS_CKPT):
            hf_hub_download(F5TTS_REPO, _ckpt_name, local_dir=F5TTS_ASSETS_DIR)
            if os.path.basename(_ckpt_name) != os.path.basename(F5TTS_CKPT):
                os.replace(os.path.join(F5TTS_ASSETS_DIR, _ckpt_name), F5TTS_CKPT)

        if not os.path.exists(F5TTS_VOCAB):
            if _vocab_name:
                hf_hub_download(F5TTS_REPO, _vocab_name, local_dir=F5TTS_ASSETS_DIR)
                if os.path.basename(_vocab_name) != os.path.basename(F5TTS_VOCAB):
                    os.replace(os.path.join(F5TTS_ASSETS_DIR, _vocab_name), F5TTS_VOCAB)
            else:
                # Some releases of this checkpoint ship the vocab under config.json
                # instead of a plain vocab.txt (see the model card).
                cfg_path = hf_hub_download(F5TTS_REPO, "config.json", local_dir=F5TTS_ASSETS_DIR)
                os.replace(cfg_path, F5TTS_VOCAB)

if not _have_both():
    raise RuntimeError(
        f"Could not obtain both {F5TTS_CKPT} and {F5TTS_VOCAB} from either the "
        f"Drive mirror or HuggingFace -- check network access and that the Drive "
        f"folder (https://drive.google.com/drive/folders/{F5TTS_DRIVE_FOLDER_ID}) "
        f"is still shared."
    )

print(f"✓ Checkpoint: {F5TTS_CKPT}")
print(f"✓ Vocab     : {F5TTS_VOCAB}")


In [ ]:
# ── L3. LOAD MODEL + synthesize_f5tts() ───────────────────────────────────────
import io
import soundfile as sf
from f5_tts.api import F5TTS

# F5TTS's real constructor param is `model` (not `model_type` -- confirmed against
# the actual installed source, src/f5_tts/api.py's F5TTS.__init__ signature has no
# model_type kwarg at all, only `model`, defaulting to "F5TTS_v1_Base"; "F5TTS_Base"
# is still a valid value -- it has its own explicit branch in that same __init__ --
# just under the right keyword).
f5tts_model = F5TTS(model="F5TTS_Base", ckpt_file=F5TTS_CKPT, vocab_file=F5TTS_VOCAB, device="cuda")

# Reference clip for zero-shot cloning -- ~6-10s clean, single-speaker Vietnamese audio
# plus its exact transcript. Upload the wav to F5TTS_ASSETS_DIR/reference.wav and fill
# in REF_TEXT_F5TTS below (a training sample from one of your RVC speakers works fine).
REF_AUDIO_F5TTS = f"{F5TTS_ASSETS_DIR}/reference.wav"
REF_TEXT_F5TTS  = ""  # <-- set this to the exact transcript of REF_AUDIO_F5TTS


def synthesize_f5tts(gen_text: str, ref_audio_path: str = None, ref_text: str = None,
                      speed: float = 1.0) -> bytes:
    """Zero-shot Vietnamese TTS via F5-TTS-Vietnamese-ViVoice. Returns WAV bytes."""
    ref_audio_path = ref_audio_path or REF_AUDIO_F5TTS
    ref_text = ref_text if ref_text is not None else REF_TEXT_F5TTS
    if not os.path.exists(ref_audio_path):
        raise FileNotFoundError(f"F5-TTS reference clip not found: {ref_audio_path}")
    if not ref_text:
        raise ValueError("REF_TEXT_F5TTS is empty -- set it to the reference clip's exact transcript.")

    wav, sr, _ = f5tts_model.infer(ref_file=ref_audio_path, ref_text=ref_text,
                                    gen_text=gen_text, speed=speed)
    buf = io.BytesIO()
    sf.write(buf, wav, sr, format="WAV")
    return buf.getvalue()


print("\u2713 synthesize_f5tts() ready.",
      "Set REF_TEXT_F5TTS and upload reference.wav before calling it." if not REF_TEXT_F5TTS else "")


In [ ]:
# ── M. LOAD PHOWHISPER (VIETNAMESE ASR) ──────────────────────────────────────
import gc
import torch
from transformers import pipeline

ASR_MODEL_NAME = "vinai/PhoWhisper-large"   # swap for -small / -medium to trade accuracy for latency
ASR_CHUNK_S    = 30

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print(f"Loading ASR model: {ASR_MODEL_NAME} ...")
try:
    asr_pipeline = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL_NAME,
        chunk_length_s=ASR_CHUNK_S,
        device="cuda" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        model_kwargs={"use_safetensors": False, "low_cpu_mem_usage": True},
    )
    print("✓ PhoWhisper ready.")
except Exception as e:
    asr_pipeline = None
    print(f"⚠️ Failed to load PhoWhisper: {e} — /transcribe will return 503 until this is fixed.")


In [ ]:
# ── I. FLASK SERVER — implements the API contract expected by voice/rvc_client.py
import io, os, tempfile, time, uuid, zipfile
from flask import Flask, request, jsonify, send_file

server = Flask(__name__)


@server.get("/health")
def health():
    return jsonify({"status": "ok"})


@server.get("/models")
def models_route():
    return jsonify({"speakers": list_trained_speakers()})


@server.get("/models/<speaker_id>/download")
def download_model_route(speaker_id):
    """Zips the trained .pth+.index for this speaker so the app can back it up locally."""
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    if not (os.path.exists(pth_path) and os.path.exists(index_path)):
        return jsonify({"error": f"Khong tim thay model da huan luyen cho {speaker_id}"}), 404

    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pth_path, arcname=f"{speaker_id}.pth")
        zf.write(index_path, arcname=f"{speaker_id}.index")
    buf.seek(0)
    return send_file(buf, mimetype="application/zip", as_attachment=True,
                      download_name=f"{speaker_id}.zip")


@server.delete("/models/<speaker_id>")
def delete_model_route(speaker_id):
    """Deletes a trained speaker's .pth/.index from Drive."""
    pth_path   = os.path.join(MODELS_DIR, f"{speaker_id}.pth")
    index_path = os.path.join(MODELS_DIR, f"{speaker_id}.index")
    removed = False
    for p in (pth_path, index_path):
        if os.path.exists(p):
            os.remove(p)
            removed = True
    with _jobs_lock:
        TRAIN_JOBS.pop(speaker_id, None)
    return jsonify({"status": "ok", "removed": removed})


@server.post("/transcribe")
def transcribe_route():
    """
    Speech-to-Text via PhoWhisper (Vietnamese-tuned) -- input half of the voice loop
    (see /baseline/f5tts and /convert below for the output half).
    multipart: audio (any format ffmpeg/librosa can decode), language (optional, e.g. "vi").
    """
    if "audio" not in request.files:
        return jsonify({"error": "thieu audio"}), 400
    if asr_pipeline is None:
        return jsonify({"error": "Mo hinh ASR chua san sang"}), 503

    language   = request.form.get("language") or None
    audio_file = request.files["audio"]
    suffix     = os.path.splitext(audio_file.filename or "")[1] or ".webm"

    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
        audio_file.save(f.name)
        tmp_path = f.name

    try:
        kwargs = {"ignore_warning": True}
        if language:
            kwargs["generate_kwargs"] = {"language": language}
        result = asr_pipeline(tmp_path, **kwargs)
        return jsonify({
            "text": result["text"].strip(),
            "language": language or "vi",
            "engine": f"phowhisper:{ASR_MODEL_NAME.rsplit('/', 1)[-1]}",
        })
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass


@server.post("/baseline/f5tts")
def baseline_f5tts_route():
    """
    Zero-shot Vietnamese TTS baseline (thesis RQ2: TTS+RVC vs zero-shot cloning).
    multipart: gen_text (required), ref_audio (wav, optional -- defaults to
    REF_AUDIO_F5TTS), ref_text (optional -- defaults to REF_TEXT_F5TTS), speed.
    """
    gen_text = request.form.get("gen_text")
    if not gen_text:
        return jsonify({"error": "thieu gen_text"}), 400

    speed = float(request.form.get("speed", 1.0))
    ref_text = request.form.get("ref_text") or None

    ref_audio_path = REF_AUDIO_F5TTS
    tmp_ref = None
    if "ref_audio" in request.files:
        tmp_ref = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        request.files["ref_audio"].save(tmp_ref.name)
        ref_audio_path = tmp_ref.name

    try:
        wav_bytes = synthesize_f5tts(gen_text, ref_audio_path=ref_audio_path,
                                      ref_text=ref_text, speed=speed)
        return send_file(io.BytesIO(wav_bytes), mimetype="audio/wav",
                          as_attachment=False, download_name="f5tts.wav")
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        if tmp_ref:
            try:
                os.unlink(tmp_ref.name)
            except OSError:
                pass


@server.post("/train")
def train_route():
    speaker_id = request.form.get("speaker_id")
    files = request.files.getlist("files")
    if not speaker_id or not files:
        return jsonify({"status": "error", "message": "Thieu speaker_id hoac file mau."}), 400

    with _jobs_lock:
        existing_status = TRAIN_JOBS.get(speaker_id, {}).get("status")
    if existing_status in ("queued", "running"):
        return jsonify({"status": "error", "message": "Giong noi nay dang duoc xu ly."}), 400

    raw_dir = tempfile.mkdtemp(prefix=f"raw_{speaker_id}_")
    for f in files:
        f.save(os.path.join(raw_dir, f.filename or f"{uuid.uuid4()}.wav"))

    _set_job(speaker_id, status="queued", message="Dang cho trong hang doi huan luyen.")
    _train_queue.put((speaker_id, raw_dir))
    return jsonify({"status": "queued", "message": "Da them vao hang doi huan luyen."})


@server.get("/train_status/<speaker_id>")
def train_status_route(speaker_id):
    with _jobs_lock:
        job = TRAIN_JOBS.get(speaker_id)
    if not job:
        return jsonify({"status": "unknown"})
    return jsonify(job)


@server.post("/convert")
def convert_route():
    speaker_id = request.form.get("speaker_id")
    if "audio" not in request.files or not speaker_id:
        return jsonify({"error": "thieu audio hoac speaker_id"}), 400

    if not has_trained_model(speaker_id):
        return jsonify({"error": f"Chua co model da huan luyen cho {speaker_id}"}), 404

    pitch      = int(request.form.get("pitch", PITCH))
    index_rate = float(request.form.get("index_rate", INDEX_RATE))

    audio_bytes = request.files["audio"].read()
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as fin:
        fin.write(audio_bytes)
        in_path = fin.name
    out_path = in_path.replace(".mp3", "_rvc.wav")

    try:
        convert_speaker(speaker_id, in_path, out_path, pitch, index_rate)
        return send_file(out_path, mimetype="audio/wav", as_attachment=False)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        try:
            os.unlink(in_path)
        except OSError:
            pass


@server.post("/stt_train")
def stt_train_route():
    adapter_id = request.form.get("adapter_id")
    base_model = request.form.get("base_model")
    transcripts_raw = request.form.get("transcripts")
    files = request.files.getlist("files")
    resume_file = request.files.get("resume_adapter")

    if not adapter_id or not base_model or not files or not transcripts_raw:
        return jsonify({"status": "error", "message": "Thieu adapter_id/base_model/files/transcripts."}), 400

    try:
        transcripts = json.loads(transcripts_raw)
    except Exception:
        return jsonify({"status": "error", "message": "transcripts khong phai JSON hop le."}), 400

    with _stt_jobs_lock:
        existing_status = STT_JOBS.get(adapter_id, {}).get("status")
    if existing_status in ("queued", "running"):
        return jsonify({"status": "error", "message": "Adapter nay dang duoc xu ly."}), 400

    raw_dir = tempfile.mkdtemp(prefix=f"stt_raw_{adapter_id}_")
    for f in files:
        f.save(os.path.join(raw_dir, f.filename or f"{uuid.uuid4()}.wav"))

    resume_zip_path = None
    if resume_file:
        resume_zip_path = os.path.join(raw_dir, "_resume.zip")
        resume_file.save(resume_zip_path)

    _set_stt_job(adapter_id, status="queued", message="Dang cho trong hang doi huan luyen.")
    _stt_train_queue.put((adapter_id, raw_dir, base_model, transcripts, resume_zip_path))
    return jsonify({"status": "queued", "message": "Da them vao hang doi huan luyen."})


@server.get("/stt_train_status/<adapter_id>")
def stt_train_status_route(adapter_id):
    with _stt_jobs_lock:
        job = STT_JOBS.get(adapter_id)
    if not job:
        return jsonify({"status": "unknown"})
    return jsonify(job)


@server.get("/stt_train/<adapter_id>/download")
def stt_train_download_route(adapter_id):
    adapter_dir = os.path.join(STT_ADAPTERS_DIR, str(adapter_id))
    weights_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    config_path = os.path.join(adapter_dir, "adapter_config.json")
    if not (os.path.exists(weights_path) and os.path.exists(config_path)):
        return jsonify({"error": f"Khong tim thay adapter da huan luyen cho {adapter_id}"}), 404

    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(weights_path, arcname="adapter_model.safetensors")
        zf.write(config_path, arcname="adapter_config.json")
    buf.seek(0)
    return send_file(buf, mimetype="application/zip", as_attachment=True,
                      download_name=f"{adapter_id}_adapter.zip")


# _run_server() runs in a daemon thread -- confirmed for real that if server.run()
# fails to bind (this cell already ran once this session, so a stale server from that
# earlier run is still holding the port), the OSError only ever surfaces as stderr
# noise from that background thread. The main cell kept going regardless and printed
# success unconditionally, so the *previous*, possibly-incomplete server silently stayed
# the one actually serving traffic through the tunnel -- misleading in exactly the way
# that cost real debugging time here. Catching the error and surfacing it as a real
# exception in the main cell (instead of a background-thread traceback) turns that into
# an immediate, actionable failure.
_server_error = []

def _run_server():
    try:
        server.run(host="0.0.0.0", port=SERVER_PORT, debug=False)
    except OSError as e:
        _server_error.append(e)


threading.Thread(target=_run_server, daemon=True).start()
time.sleep(1.5)
if _server_error:
    raise RuntimeError(
        f"Failed to bind port {SERVER_PORT}: {_server_error[0]} -- this cell was already run "
        f"once this session; a stale server from that earlier run is still bound and is what's "
        f"actually serving traffic (silently missing whatever this run would have added/fixed). "
        f"Re-running just this cell will NOT fix it -- Runtime > Disconnect and delete runtime, "
        f"then Run all exactly once."
    )
print(f"✓ Voice server listening on port {SERVER_PORT}")
print("  Routes: /health  /models  /transcribe  /baseline/f5tts  /train  /train_status/<id>  /convert")
print("  STT Tier 2:  /stt_train  /stt_train_status/<id>  /stt_train/<id>/download")


In [ ]:
# ── J. CLOUDFLARED TUNNEL — exposes the server publicly over HTTPS ──────────
import os, re, subprocess, threading, time

if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
    print("✓ cloudflared downloaded.")
else:
    print("✓ cloudflared already present.")

_tunnel_url = []


def _run_tunnel():
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://localhost:{SERVER_PORT}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
        if m and not _tunnel_url:
            _tunnel_url.append(m.group(0))


threading.Thread(target=_run_tunnel, daemon=True).start()

print("Starting cloudflared tunnel… (waiting up to 30s)")
for _ in range(30):
    if _tunnel_url:
        break
    time.sleep(1)

if not _tunnel_url:
    raise RuntimeError("Tunnel URL not detected — check cloudflared output above.")

RVC_ENDPOINT = _tunnel_url[0]
print(f"\n✓ Tunnel active: {RVC_ENDPOINT}")


In [ ]:
# ── K. COPY THIS URL INTO THE APP ─────────────────────────────────────────────
# Manager dashboard (/, after /login) -> "Ket noi RVC (Colab)" section
# DO NOT stop or restart the runtime -- cloudflared will assign a new URL and
# the app's stored endpoint will need to be updated again.

print("=" * 70)
print("  Paste this URL into the manager dashboard:")
print("  /  (after /login)  ->  section 'Ket noi RVC (Colab)'")
print("=" * 70)
print()
print(f"  {RVC_ENDPOINT}")
print()
print("Keep this notebook running -- training jobs submitted from the app are")
print("processed automatically (one at a time) as long as this session stays")
print("alive. The endpoint goes offline as soon as the runtime stops.")

import urllib.request, json, time as _time
_time.sleep(2)
try:
    with urllib.request.urlopen(f"{RVC_ENDPOINT}/health", timeout=10) as r:
        print(f"\n✓ /health check passed: {json.loads(r.read())}")
except Exception as exc:
    print(f"\n⚠️ /health check failed: {exc}")
    print("   The server may still be starting -- retry in a few seconds.")
